In [27]:
import pandas as pd
import pickle
import numpy as np
from IPython.display import Image, display


movies = pd.read_csv('movies_final_level7.csv')

with open('movie_embeddings.pkl', 'rb') as f:
    sbert_vectors = pickle.load(f)

with open("movie_tfidf.pkl", "rb") as f:
    tfidf_vectors = pickle.load(f)

print("✅ Laboratory Reloaded. Systems are light and fast.")

✅ Laboratory Reloaded. Systems are light and fast.


In [28]:
movies

,id,title,tag,vote_count,rating,crew,director
0,862.0,Toy Story (1995),johnlasseter johnlasseter johnlasseter animati...,5415.0,7.691378,johnlasseter,John Lasseter
1,8844.0,Jumanji (1995),joejohnston joejohnston joejohnston adventure ...,2413.0,6.891917,joejohnston,Joe Johnston
2,15602.0,Grumpier Old Men (1995),howarddeutch howarddeutch howarddeutch romance...,92.0,6.450950,howarddeutch,Howard Deutch
3,31357.0,Waiting to Exhale (1995),forestwhitaker forestwhitaker forestwhitaker c...,34.0,6.209113,forestwhitaker,Forest Whitaker
4,11862.0,Father of the Bride Part II (1995),charlesshyer charlesshyer charlesshyer comedy ...,173.0,5.801544,charlesshyer,Charles Shyer
...,...,...,...,...,...,...,...
11193,248705.0,The Visitors: Bastille Day (2016),jean-mariepoiré jean-mariepoiré jean-mariepoir...,167.0,4.392138,jean-mariepoiré,Jean-Marie Poiré
11194,44918.0,Titanic 2 (2010),shanevandyke shanevandyke shanevandyke action ...,55.0,4.514828,shanevandyke,Shane Van Dyke
11195,426272.0,Take Me (2017),pathealy pathealy pathealy comedy crime comedy...,38.0,6.150273,pathealy,Pat Healy
11196,432789.0,The Incredible Jessica James (2017),jimstrouse jimstrouse jimstrouse romance comed...,37.0,6.256615,jimstrouse,Jim Strouse


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
min_votes = movies["vote_count"].min()
max_votes = movies["vote_count"].max()

def norm_votes(v):
    return (v - min_votes) / (max_votes - min_votes)

In [31]:
def recommend(target_idx):
    sim_tfidf = cosine_similarity(tfidf_vectors[target_idx], tfidf_vectors).flatten()
    sim_sbert = cosine_similarity([sbert_vectors[target_idx]], sbert_vectors).flatten()
    
    combined_sim = (0.7 * sim_tfidf) + (0.3 * sim_sbert)
    
    similar_indices = sorted(
        list(enumerate(combined_sim)), 
        reverse=True, 
        key=lambda x: x[1]
    )[1:51]

    rec_list = []

    for i in similar_indices:
        idx_i = i[0]
        sim_score = i[1]
        temp_df = movies.iloc[idx_i]

        # Filter weak movies
        if temp_df["rating"] < 6.5:
            continue

        rating_score = temp_df["rating"] / 10
        popularity_score = norm_votes(temp_df["vote_count"])

        final_score = (
            0.6 * sim_score +
            0.25 * rating_score +
            0.15 * popularity_score
        )

        rec_list.append({
            "title": temp_df["title"],
            "rating": temp_df["rating"],
            "score": final_score
        })

    final_recommendations = sorted(
        rec_list, 
        key=lambda x: x["score"], 
        reverse=True
    )

    print(f"Top 10 High Quality Recommendation for {movies.loc[target_idx, 'title']}")
    print("-" * 30)

    rec = []
    for movie in final_recommendations[:10]:
        print(f"- {movie['title']} (Rating: {movie['rating']:.2f})")
        rec.append(movie["title"])

    return rec

In [32]:
def director_spotlight_by_idx(target_idx, n=5):
    target_director = movies.loc[target_idx, "crew"]

    director_movies = movies[(movies["crew"]==target_director) & (movies.index != target_idx)]

    target_director = movies.loc[target_idx, "director"]

    if director_movies.empty:
        return target_director, []

    top_director_df = director_movies.sort_values("rating", ascending = False).head(n)
    director_recs = top_director_df[["title", "rating"]].to_dict("records")

    return target_director, director_recs

In [33]:
def enhanced_recommend(movie_title):
    movie_title = movie_title.lower()

    match = movies[movies["title"].str.lower() == movie_title]

    if match.empty:
        match = movies[movies["title"].str.lower().str.contains(movie_title)]

    if match.empty:
        print("Movie not found 😅")
        return 

    target_idx = match.sort_values("rating", ascending=False).index[0]
    target_title = movies.loc[target_idx, "title"]

    print("\n" + "=" * 60)

    recommend(target_idx)

    director_name, dir_recs = director_spotlight_by_idx(target_idx)

    if dir_recs:
        print(f"\n🎥 Director's Spotlight: More from {director_name}")
        print("-" * 30)
        for d_movie in dir_recs:
            print(f"- {d_movie['title']} (Rating: {d_movie['rating']:.2f})")
    else:
        # Fixed the print here to use the pretty name variable too
        print(f"\n🎥 No other high-quality films by {director_name} in our database.")

In [34]:
movies_to_test = [
    "Finding Nemo",
    "Pulp Fiction",
    "The Avengers",
    "Inception",
    "The Dark Knight",
    "Toy Story",
    "Fight Club",
    "Interstellar",
    "Before Sunset",
    "Eternal Sunshine of the Spotless Mind",
    "Memento",
    "Taare Zameen Par",
    "Shutter Island",
    "Wolf of Wall Street",
    "Lord of the Rings",
    "Harry Potter",
    "12 Angry Men",
    "Pather Panchali",
    "Silence of the Lambs",
    "Schindler's List",
    "The Godfather",
    "prestige",
    "Incendies",
    "Psycho",
    "No Country for Old Men",
    "There will be blood",
    "Machinist"
]

In [35]:
def batch_recommend(movie_list):
    for movie in movie_list:
        print("\n" + "="*60)
        enhanced_recommend(movie)

In [36]:
batch_recommend(movies_to_test)



Top 10 High Quality Recommendation for Finding Nemo (2003)
------------------------------
- WALL·E (2008) (Rating: 7.79)
- Coraline (2009) (Rating: 7.28)
- Winnie the Pooh (2011) (Rating: 6.74)
- Monsters University (2013) (Rating: 6.99)
- Wreck-It Ralph (2012) (Rating: 7.09)
- Anastasia (1997) (Rating: 7.38)
- Professor Layton and the Eternal Diva (2009) (Rating: 7.03)
- Ponyo (2008) (Rating: 7.46)
- Azur & Asmar: The Princes' Quest (2006) (Rating: 7.18)
- Madagascar (2005) (Rating: 6.60)

🎥 Director's Spotlight: More from Andrew Stanton
------------------------------
- WALL·E (2008) (Rating: 7.79)
- John Carter (2012) (Rating: 6.10)


Top 10 High Quality Recommendation for Pulp Fiction (1994)
------------------------------
- Reservoir Dogs (1992) (Rating: 8.08)
- Snatch (2000) (Rating: 7.68)
- The Hateful Eight (2015) (Rating: 7.59)
- Kill Bill: Vol. 2 (2004) (Rating: 7.69)
- The Night of the Hunter (1955) (Rating: 7.75)
- Million Dollar Baby (2004) (Rating: 7.68)
- Ocean's Eleven 